# Data Cleaning Notebook

Superstore end-to-end analytics project notebook.

## Load raw data

In [ ]:

import pandas as pd
import numpy as np
from pathlib import Path

RAW_PATH = Path('../Dataset/Raw_Data.csv')
OUTPUT_PATH = Path('../Dataset/Cleaned_Data.csv')

def snake_case(name):
    import re
    return re.sub(r'[^0-9a-zA-Z]+', '_', name.strip().lower()).strip('_')

df = pd.read_csv(RAW_PATH, encoding='latin1')
print('Raw shape:', df.shape)
df.head()


## Cleaning operations
Missing values, duplicates, type conversion, validation, standardization.

In [ ]:

# Standardize columns and text fields
df.columns = [snake_case(c) for c in df.columns]
for c in df.select_dtypes(include='object').columns:
    df[c] = df[c].astype(str).str.strip().replace({'nan': np.nan, 'None': np.nan, '': np.nan})

# Date conversion
for c in ['order_date', 'ship_date']:
    df[c] = pd.to_datetime(df[c], errors='coerce')

# Missing value handling
for c in df.select_dtypes(include='object').columns:
    df[c] = df[c].fillna('Unknown')
for c in df.select_dtypes(include=['number']).columns:
    df[c] = df[c].fillna(df[c].median())

# Duplicates
before = len(df)
df = df.drop_duplicates()
print('Duplicates removed:', before - len(df))

# Type conversion and validation
df['postal_code'] = df['postal_code'].astype('Int64').astype(str).replace('<NA>', 'Unknown')
df['sales'] = pd.to_numeric(df['sales'], errors='coerce').fillna(0)
df['profit'] = pd.to_numeric(df['profit'], errors='coerce').fillna(0)
df['quantity'] = pd.to_numeric(df['quantity'], errors='coerce').fillna(0).astype(int)
df['discount'] = pd.to_numeric(df['discount'], errors='coerce').fillna(0).clip(0, 1)


## Outlier detection and feature engineering
Outliers are flagged, not deleted, because high-value enterprise orders are valid commercial behavior.

In [ ]:

# Outlier flags using IQR - keep valid business extremes, flag for review
for c in ['sales', 'profit', 'quantity', 'discount']:
    q1, q3 = df[c].quantile([0.25, 0.75])
    iqr = q3 - q1
    df[f'{c}_outlier_flag'] = ((df[c] < q1 - 1.5*iqr) | (df[c] > q3 + 1.5*iqr)).astype(int)

# Feature engineering
df['order_year'] = df['order_date'].dt.year
df['order_month'] = df['order_date'].dt.to_period('M').dt.to_timestamp()
df['order_month_name'] = df['order_date'].dt.strftime('%b')
df['order_quarter'] = 'Q' + df['order_date'].dt.quarter.astype(str)
df['ship_days'] = (df['ship_date'] - df['order_date']).dt.days
df.loc[df['ship_days'] < 0, 'ship_days'] = np.nan
df['ship_days'] = df['ship_days'].fillna(df['ship_days'].median()).astype(int)
df['revenue'] = df['sales']
df['cost'] = df['sales'] - df['profit']
df['profit_margin'] = np.where(df['sales'] != 0, df['profit'] / df['sales'], 0)
df['discount_band'] = pd.cut(df['discount'], [-0.01,0,0.1,0.2,0.5,1], labels=['No Discount','Low','Medium','High','Very High'])
df['profitability_status'] = np.where(df['profit'] >= 0, 'Profitable', 'Loss-Making')
df['customer_order_count'] = df.groupby('customer_id')['order_id'].transform('nunique')
df['customer_lifetime_value'] = df.groupby('customer_id')['profit'].transform('sum')
df['customer_value_segment'] = pd.qcut(df['customer_lifetime_value'].rank(method='first'), 4, labels=['Low Value','Medium Value','High Value','VIP'])
df.to_csv(OUTPUT_PATH, index=False)
print('Cleaned file saved:', OUTPUT_PATH)
df.head()
